In [9]:
import pandas as pd
import numpy as np

In [10]:
df = pd.read_pickle(r"D:\2026_Summer\TradingApp\data\v3.0\all_symbol_min_full_main_close_k_1.pkl")

In [15]:
# ============================================================
# 1. Collapse minute bars -> daily volume / turnover per symbol
#    (trading_date already folds the night session into the
#     correct session, so grouping on it is the right key)
# ============================================================
sym = df['underlying_symbol'].astype('category')      # categorical key => much faster groupby
daily = (df[['volume', 'total_turnover']]
           .groupby([sym, df['trading_date']], observed=True)
           .sum())
daily.index.names = ['symbol', 'date']
daily = daily.reset_index()

all_days = np.sort(daily['date'].unique())            # union of every trading day in the file
last_day = pd.Timestamp(all_days[-1])

In [16]:
# ============================================================
# 2. Window stats.
#    turnover_* = mean 亿元/session, denominator = ALL sessions in the
#                 window, so dead symbols decay toward 0 (that's the point)
#    med_*      = median over sessions the symbol actually traded --
#                 immune to one blow-off day faking liquidity
#    cover_*    = fraction of the window's sessions with data
# ============================================================
def window_stats(days_back=None, suffix=''):
    if days_back is None:
        sub, n = daily, len(all_days)
    else:
        n = min(days_back, len(all_days))
        sub = daily[daily['date'] >= all_days[-n]]
    g = sub.groupby('symbol', observed=True)
    return pd.DataFrame({
        f'turnover{suffix}':     g['total_turnover'].sum() / n / 1e8,
        f'med_turnover{suffix}': g['total_turnover'].median() / 1e8,
        f'vol{suffix}':          g['volume'].sum() / n,
        f'cover{suffix}':        g.size() / n,
    })

life = daily.groupby('symbol', observed=True)['date'].agg(first_date='min', last_date='max')

summary = (life
           .join(window_stats(None, '_all'))
           .join(window_stats(250,  '_1y'))
           .join(window_stats(60,   '_3m'))
           .fillna(0))
summary['dead_days'] = (last_day - summary['last_date']).dt.days

In [18]:
# ============================================================
# 3. Ranking: average daily turnover, high -> low
# ============================================================
rank = summary.sort_values('turnover_1y', ascending=False)

show = rank[['turnover_1y', 'med_turnover_1y', 'turnover_3m', 'turnover_all',
             'cover_1y', 'vol_1y', 'first_date', 'last_date', 'dead_days']].copy()
for c in ['turnover_1y', 'med_turnover_1y', 'turnover_3m', 'turnover_all']:
    show[c] = show[c].round(2)
show['cover_1y'] = show['cover_1y'].round(2)
show['vol_1y']   = show['vol_1y'].round(0)

pd.set_option('display.max_rows', 100)
print(f'{len(summary)} symbols | data through {last_day.date()} | turnover in 亿元/session')

# Ordered list for manual filtering
order = list(rank.index)
print(order)

DROP = []                                   # <- paste the ones you want out
KEEP = [s for s in order if s not in DROP]
len(KEEP), KEEP

79 symbols | data through 2026-07-29 | turnover in 亿元/session
['AU', 'AG', 'IC', 'T', 'IF', 'TS', 'JM', 'TF', 'SN', 'CU', 'SC', 'P', 'RU', 'NI', 'FG', 'RB', 'M', 'AL', 'V', 'TA', 'IH', 'MA', 'SA', 'CF', 'Y', 'OI', 'I', 'LH', 'PP', 'FU', 'EB', 'L', 'EG', 'HC', 'ZN', 'SR', 'JD', 'BU', 'SP', 'C', 'SS', 'RM', 'NR', 'PG', 'AP', 'A', 'UR', 'SM', 'CJ', 'SF', 'PF', 'B', 'J', 'PB', 'LU', 'PK', 'CS', 'BC', 'CY', 'RR', 'FB', 'BB', 'WR', 'RS', 'AO', 'BR', 'LC', 'EC', 'JR', 'LR', 'IM', 'SI', 'SH', 'PX', 'RI', 'PM', 'TL', 'WH', 'ZC']


(79,
 ['AU',
  'AG',
  'IC',
  'T',
  'IF',
  'TS',
  'JM',
  'TF',
  'SN',
  'CU',
  'SC',
  'P',
  'RU',
  'NI',
  'FG',
  'RB',
  'M',
  'AL',
  'V',
  'TA',
  'IH',
  'MA',
  'SA',
  'CF',
  'Y',
  'OI',
  'I',
  'LH',
  'PP',
  'FU',
  'EB',
  'L',
  'EG',
  'HC',
  'ZN',
  'SR',
  'JD',
  'BU',
  'SP',
  'C',
  'SS',
  'RM',
  'NR',
  'PG',
  'AP',
  'A',
  'UR',
  'SM',
  'CJ',
  'SF',
  'PF',
  'B',
  'J',
  'PB',
  'LU',
  'PK',
  'CS',
  'BC',
  'CY',
  'RR',
  'FB',
  'BB',
  'WR',
  'RS',
  'AO',
  'BR',
  'LC',
  'EC',
  'JR',
  'LR',
  'IM',
  'SI',
  'SH',
  'PX',
  'RI',
  'PM',
  'TL',
  'WH',
  'ZC'])